# 05 — Giai đoạn 5: Safe Inference (guard + MC Dropout trên LoRA) (Colab)

Thiết kế: `docs/safe_inference.md`. Lớp an toàn đứng **sau** model (không đổi model/prompt/retrieval) nên chạy được
trên các run đã có, không cần sinh lại.

| Bước | Việc | Chạy khi nào | GPU | Chi phí OpenAI |
|---|---|---|---|---|
| 1 | Đo guard trên SFT v2 + dev | **ngay** | không | 0 |
| 2 | Guard trên Qwen gốc (dev) → chấm → so sánh | sau bước 0 notebook 04 | không | ~0,1 USD (chỉ câu bị đổi) |
| 3 | Tín hiệu deterministic trên Qwen gốc (dev) → hiệu chỉnh | sau bước 0 notebook 04 | ~5 phút | 0 |
| 4 | Thử pipeline MC Dropout (adapter SFT v1, 5 case) | **ngay** | ~3 phút | 0 |
| 5 | MC Dropout trên model cuối (SFT v3/DPO) → hiệu chỉnh → áp dụng → chấm | sau notebook 04 | ~10–20 phút | ~0,2 USD |
| 6 | Nghiệm thu cuối trên test (MỘT lần) | sau khi chốt cấu hình | ~1 giờ | ~1 USD |

Qwen gốc ở đây là run **có reranker** do bước 0 của notebook 04 sinh ra (cùng protocol truy xuất với
SFT v2/DPO). Dùng chung thư mục Drive `KLTN/gout_llmops_work` và cache judge với notebook 04. Không chạy song song với notebook 04
trên hai tài khoản.

## 0. Cấu hình

In [ ]:
REPO_URL = "https://github.com/NtQuang-23521287/Developing_A_Vietnamese_Gout_Chatbot_Based_On_Large_Language_Models.git"
BRANCH = "main"                                         # nhánh chứa code Giai đoạn 5
KLTN_FOLDER_ID = "1ptzef1HYRBwtKzHAFXW1e4cQ_9xhyKb4"   # lấy từ URL drive.google.com/drive/folders/<ID>
DRIVE_SUBDIR = "gout_llmops_work"

BASE_NAME = "qwen3_8b_base_rerank"   # Qwen3-8B gốc có reranker, sinh + chấm ở bước 0 notebook 04
# Run SFT v1 + adapter (lora_dropout = 0) — CHỈ để thử pipeline MC Dropout ở bước 4
SFT_V1_RUN = ("artifacts/runs/20261031T092014Z_baseline_qwen3_8b_sft_lr5e5_10874_generation_381/"
              "artifacts/runs/20261031T092014Z_baseline_qwen3_8b_sft_lr5e5_10874")
SFT_V1_ADAPTER = "artifacts/runs/sft_qwen3_8b_v2/checkpoint-150"

# Bước 5–6: model cuối do notebook 04 chọn — "qwen3_8b_dpo_v1" hoặc "qwen3_8b_sft_v3"
FINAL_NAME = "qwen3_8b_dpo_v1"
FINAL_ADAPTER = "artifacts/runs/dpo_qwen3_8b_v1"        # hoặc artifacts/runs/sft_qwen3_8b_v3
N_SAMPLES = 10                                          # số lượt MC Dropout
SAFE_TAG = "safe_v1"

## 1. GPU, Google Drive, secrets

In [ ]:
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip().splitlines()
print("GPU:", gpu[0] if gpu else "KHÔNG CÓ GPU — Runtime > Change runtime type")
GPU_MEM_GB = int(gpu[0].split(",")[-1]) / 1024 if gpu else 0

from google.colab import drive, userdata
drive.mount("/content/drive")   # vừa thêm lối tắt KLTN sau khi mount? dùng drive.mount("/content/drive", force_remount=True)

# Thư mục được chia sẻ hiện ở .shortcut-targets-by-id/<ID>/ (tìm theo ID, không phụ thuộc lối tắt đặt ở đâu)
import glob
_cands = (glob.glob(f"/content/drive/.shortcut-targets-by-id/{KLTN_FOLDER_ID}/*")
          + ["/content/drive/MyDrive/KLTN"]
          + glob.glob("/content/drive/.shortcut-targets-by-id/*/KLTN"))
KLTN_DIR = next((p for p in _cands if os.path.isdir(p)), None)
assert KLTN_DIR, ("Không thấy thư mục KLTN. Trên Google Drive: chuột phải KLTN > Sắp xếp > Thêm lối tắt > Drive của tôi, "
                  "rồi chạy lại ô này với drive.mount('/content/drive', force_remount=True)")
DRIVE_DIR = f"{KLTN_DIR}/{DRIVE_SUBDIR}"
try:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    with open(f"{DRIVE_DIR}/.write_test", "w") as f:
        f.write("ok")
    os.remove(f"{DRIVE_DIR}/.write_test")
except OSError as e:
    raise PermissionError(f"Không ghi được vào {DRIVE_DIR}: cần quyền Người chỉnh sửa trên thư mục KLTN") from e
print("DRIVE_DIR =", DRIVE_DIR)

for key in ("OPENAI_API_KEY", "HF_TOKEN"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f"[!] Chưa có secret {key}")
assert os.environ.get("OPENAI_API_KEY"), "Cần OPENAI_API_KEY cho answer/filter/judge"

## 2. Lấy code và cài thư viện

In [ ]:
%cd /content
if not os.path.isdir("/content/repo"):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} {REPO_URL} repo
else:
    !git -C /content/repo pull --ff-only
%cd /content/repo
!git log --oneline -1

!pip install -q peft bitsandbytes accelerate "openai>=1.50" pyyaml pandas matplotlib
!pip install -q -e . --no-deps

# Luôn thấy gói gout_llmops (cả trong kernel lẫn các lệnh !python), kể cả khi pip install -e ở trên không thành công
import sys
SRC = "/content/repo/src"
if SRC not in os.environ.get("PYTHONPATH", "").split(":"):
    os.environ["PYTHONPATH"] = ":".join(p for p in (SRC, os.environ.get("PYTHONPATH", "")) if p)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
!python --version
!python -c "import gout_llmops; print('gout_llmops OK:', gout_llmops.__file__)"

import importlib.metadata as md
for p in ("torch", "transformers", "peft", "bitsandbytes"):
    try:
        print(f"{p:>14} {md.version(p)}")
    except md.PackageNotFoundError:
        print(f"{p:>14} (chưa cài)")

## 3. Lưu trữ trên Drive + kiểm tra nhanh
Giống notebook 04: `artifacts/judge_cache` trỏ thẳng lên Drive (không chấm lại case đã chấm), các thư mục kết quả
đồng bộ bằng `save()` / `restore()`.

In [ ]:
import json, shutil

LINKED = ["artifacts/sft_v2", "artifacts/judge_cache", "artifacts/dpo_v1"]
SYNCED = ["data/sft_v2", "data/dpo", "artifacts/runs", "artifacts/reports"]

def link_to_drive(p):
    dst = f"{DRIVE_DIR}/{p}"
    if os.path.islink(p):
        return
    if os.path.isdir(dst):   # đã có từ phiên trước: chỉ bổ sung file Drive chưa có, không ghi đè
        if os.path.isdir(p):
            subprocess.run(["rsync", "-a", "--ignore-existing", f"{p}/", f"{dst}/"], check=True)
    elif os.path.isdir(p):
        shutil.copytree(p, dst)
    else:
        os.makedirs(dst)
    shutil.rmtree(p, ignore_errors=True)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    os.symlink(dst, p)

def save():
    for p in SYNCED:
        if os.path.exists(p):
            os.makedirs(f"{DRIVE_DIR}/{p}", exist_ok=True)
            subprocess.run(["rsync", "-a", f"{p}/", f"{DRIVE_DIR}/{p}/"], check=True)
    print("Đã lưu lên", DRIVE_DIR)

def restore():
    for p in SYNCED:
        if os.path.isdir(f"{DRIVE_DIR}/{p}"):
            os.makedirs(p, exist_ok=True)
            subprocess.run(["rsync", "-a", f"{DRIVE_DIR}/{p}/", f"{p}/"], check=True)

def latest_run(name):
    return open(f"artifacts/runs/LATEST_{name}.txt").read().strip()

for p in LINKED:
    link_to_drive(p)
restore()
!python -m pytest -q tests/test_safety.py

## Bước 1 — Đo guard (CPU)
Recall/báo nhầm trên câu hỏi SFT v2 (nhãn LLM, nhiễu) và dev. Split test KHÔNG dùng ở đây (bước 6).

In [ ]:
!python scripts/safe_inference.py guard-eval
save()

## Bước 2 — Chỉ guard trên Qwen gốc (dev) → chấm → so sánh
Chưa có độ bất định (`--calibration none`): chỉ chèn banner red flag / lưu ý tự ý dùng thuốc. Judge chỉ gọi API cho
case bị đổi (cache theo input). Dòng *Kiểm tra* trong báo cáo phải là 0 case lệch điểm.

In [ ]:
BASE_RUN = latest_run(BASE_NAME)
!python scripts/safe_inference.py apply --run {BASE_RUN} --split dev --calibration none --tag guard_v1 --overwrite
GUARD_RUN = f"{BASE_RUN}__guard_v1"
!python scripts/eval_adapter.py score --run {GUARD_RUN} --split dev
!python scripts/safe_inference.py report --base {BASE_RUN} --safe {GUARD_RUN}
!python scripts/compare_runs_v2.py {BASE_RUN} {GUARD_RUN} --names Qwen-Base Qwen-Base+Guard --split dev --out artifacts/reports/safety/base_vs_guard_dev
save()

## Bước 3 — Tín hiệu deterministic trên Qwen gốc (dev)
Model gốc không có LoRA nên chỉ có `det_*` (NLL/entropy của chính câu trả lời) và `ret_low_relevance`. Đây là mốc
để xem MC Dropout ở bước 5 có thêm thông tin gì so với tín hiệu một lượt hay không. Ghi dần, bị ngắt thì chạy lại ô.

In [ ]:
BASE_RUN = latest_run(BASE_NAME)
!python scripts/safe_inference.py uncertainty --run {BASE_RUN} --split dev --adapter none --n-samples 0
!python scripts/safe_inference.py calibrate --run {BASE_RUN} --split dev --plot
save()

## Bước 4 — Thử pipeline MC Dropout (adapter SFT v1, 5 case)
Adapter SFT v1 train với `lora_dropout = 0`, nên phải chèn dropout lúc suy luận (`--p-override 0.05`). Đây **không
phải** MC Dropout đúng nghĩa: chỉ để kiểm tra code chạy trên GPU (VRAM, tốc độ, khớp số token prompt). Không dùng số
liệu này. Chạy trên bản sao của run để không ghi vào run SFT v1.

In [ ]:
!git lfs install --skip-repo -q && git lfs pull --include "{SFT_V1_ADAPTER}/*"
!rm -rf /content/smoke_run && cp -r {SFT_V1_RUN} /content/smoke_run
!python scripts/safe_inference.py uncertainty --run /content/smoke_run --split dev --model-config configs/models/qwen3_8b.yaml --adapter {SFT_V1_ADAPTER} --n-samples 4 --p-override 0.05 --limit 5

rows = [json.loads(l) for l in open("/content/smoke_run/safety/uncertainty.jsonl")]
for r in rows:
    print(r["case_id"], {k: round(v, 4) if isinstance(v, float) else v for k, v in r.items() if k != "case_id"})
assert all("error" not in r for r in rows), "có case lỗi"
assert all(r["prompt_tokens_match"] for r in rows), "prompt dựng lại lệch với lúc sinh"
assert all(r["mc_nll_std"] > 0 for r in rows), "dropout không có tác dụng (mc_nll_std = 0)"
print("Pipeline MC Dropout OK")

## Bước 5 — MC Dropout trên model cuối (sau notebook 04)
Chạy khi notebook 04 đã sinh + chấm run dev của model cuối (`LATEST_{FINAL_NAME}.txt`). Adapter phải có
`lora_dropout > 0` (SFT v3 / DPO v1: 0.05).

1. `uncertainty`: N lượt MC Dropout cho câu trả lời cuối của 114 case dev.
2. `calibrate`: AUROC + CI95 từng tín hiệu, đường risk–coverage, chọn tín hiệu và τ (ngân sách từ chối 15%). Nhánh
   từ chối chỉ BẬT khi cận dưới CI95 AUROC > 0,5.
3. `apply` → `score` → `report` + `compare_runs_v2` (Base / model cuối / model cuối + an toàn).

In [ ]:
BASE_RUN = latest_run(BASE_NAME)
FINAL_RUN = latest_run(FINAL_NAME)
print("FINAL_RUN =", FINAL_RUN)
!python scripts/safe_inference.py uncertainty --run {FINAL_RUN} --split dev --n-samples {N_SAMPLES}
!python scripts/safe_inference.py calibrate --run {FINAL_RUN} --split dev --plot
save()

In [ ]:
import os
CAL = f"artifacts/reports/safety/calibration_{os.path.basename(FINAL_RUN)}_dev.json"
cal = json.load(open(CAL))
print("Tín hiệu:", cal["signal"], "| τ =", round(cal["threshold"], 4), "| bật từ chối:", cal["enabled"], "|", cal["reason"])

!python scripts/safe_inference.py apply --run {FINAL_RUN} --split dev --calibration {CAL} --tag {SAFE_TAG} --overwrite
SAFE_RUN = f"{FINAL_RUN}__{SAFE_TAG}"
!python scripts/eval_adapter.py score --run {SAFE_RUN} --split dev
!python scripts/safe_inference.py report --base {FINAL_RUN} --safe {SAFE_RUN}
!python scripts/compare_runs_v2.py {BASE_RUN} {FINAL_RUN} {SAFE_RUN} --names Qwen-Base Final Final+Safe --split dev --out artifacts/reports/safety/final_vs_safe_dev
save()

Đọc kết quả:
- `report`: nhóm **abstain_uncertain** thường có reward thấp hơn (mất completeness) nhưng tỉ lệ safety/hallucination ≥ 2
  phải giảm. Đó là đánh đổi cần trình bày cùng đường risk–coverage (`calibration_*.png`), không chỉ nhìn reward.
- Nếu calibration TẮT (AUROC không hơn ngẫu nhiên) thì run an toàn chỉ có guard: đó là kết quả trung thực về MC Dropout
  trên dữ liệu này, ghi vào khóa luận như vậy.
- Thử `uncertainty_action: hedge` trong `configs/safety/safe_inference_v1.yaml` (đổi `--tag`) để so sánh với abstain.

## Bước 6 — Nghiệm thu cuối trên test (CHẠY MỘT LẦN)
Chỉ chạy khi mọi lựa chọn (model, tín hiệu, τ, luật guard, mẫu câu) đã chốt trên dev. τ dùng lại từ calibration
**dev**; `calibrate --split test` chỉ để báo cáo AUROC ngoài mẫu.

In [ ]:
RUN_TEST_ONCE = False   # đổi thành True khi đã chốt cấu hình
assert RUN_TEST_ONCE, "Chưa chốt cấu hình — không chạy test"

!python scripts/safe_inference.py guard-eval --final-test
# Qwen gốc trên test, cùng retriever (có reranker)
!python scripts/eval_adapter.py generate --name {BASE_NAME}_test --split test --final-test
BASE_TEST_RUN = latest_run(f"{BASE_NAME}_test")
!python scripts/eval_adapter.py score --run {BASE_TEST_RUN} --split test
!python scripts/eval_adapter.py generate --name {FINAL_NAME}_test --adapter {FINAL_ADAPTER} --split test --final-test
FINAL_TEST_RUN = latest_run(f"{FINAL_NAME}_test")
!python scripts/eval_adapter.py score --run {FINAL_TEST_RUN} --split test
!python scripts/safe_inference.py uncertainty --run {FINAL_TEST_RUN} --split test --n-samples {N_SAMPLES} --final-test
!python scripts/safe_inference.py calibrate --run {FINAL_TEST_RUN} --split test --plot --final-test
!python scripts/safe_inference.py apply --run {FINAL_TEST_RUN} --split test --calibration {CAL} --tag {SAFE_TAG} --final-test --overwrite
SAFE_TEST_RUN = f"{FINAL_TEST_RUN}__{SAFE_TAG}"
!python scripts/eval_adapter.py score --run {SAFE_TEST_RUN} --split test
!python scripts/safe_inference.py report --base {FINAL_TEST_RUN} --safe {SAFE_TEST_RUN}
!python scripts/compare_runs_v2.py {BASE_TEST_RUN} {FINAL_TEST_RUN} {SAFE_TEST_RUN} --names Qwen-Base Final Final+Safe --split test --out artifacts/reports/safety/final_vs_safe_test
save()

## Sau khi chạy xong
Mang về repo và commit: `artifacts/reports/safety/` (guard_eval, calibration_*.json/.md/.png, safety_report_*,
so sánh), các run `*__guard_v1`, `*__safe_v1` (predictions + safety_decisions + safe_manifest + eval_v2), và
`<run>/safety/uncertainty*.json*` của các run đã chấm độ bất định.